# 07 — Misclassification analysis (Figure 7)

**Tujuan.** Siapkan metadata kesalahan dari final-test JSON yang sudah dibekukan, tanpa inference ulang. Figure 7 hlm. 91512–91513 menunjukkan DDR DR→No DR (FN) dan No DR→DR (FP); ini semantik biner, berbeda dari salah grade lima kelas. Gambar test **tidak dibuka** saat readiness.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
from reproduction.cadvit.visualization import load_controlled_result, error_records
if ARTIFACT_ROOT:
    item = load_controlled_result(ARTIFACT_ROOT, "balanced_v1", "ddr_full")
    print("DDR", item["status"])
    if item["status"] == "AVAILABLE":
        errors = error_records(item["result"])
        print(errors["status"], "error records", len(errors.get("errors", [])),
              "task", errors.get("task"))
        binary = error_records(item["result"], binary_projection=True)
        print("binary DR/No DR projection (ASSUMED)", binary["status"],
              "FP/FN records", len(binary.get("errors", [])))
        print("Gallery PENDING: explicit post-evaluation image opt-in required")
else:
    print("PENDING: frozen per-image prediction metadata unavailable")

PENDING: frozen per-image prediction metadata unavailable


In [3]:
from reproduction.cadvit.visualization import plot_error_gallery, save_campaign_figure
if os.environ.get("CAD_VIT_SHOW_TEST_GALLERY") == "1":
    from IPython.display import display
    if ARTIFACT_ROOT is None:
        raise ValueError("CAD_VIT_ARTIFACT_ROOT required")
    item = load_controlled_result(ARTIFACT_ROOT, "balanced_v1", "ddr_full")
    if item["status"] != "AVAILABLE":
        raise ValueError("Frozen DDR final-test JSON unavailable or unverified")
    rows = error_records(item["result"], binary_projection=True)
    if rows["status"] != "AVAILABLE":
        raise ValueError("Per-image prediction metadata unavailable")
    dataset_root = os.environ["CAD_VIT_DATA_ROOT"]
    figure = plot_error_gallery(rows["errors"], dataset_root, allow_test_images=True)
    display(figure)
    if os.environ.get("CAD_VIT_SAVE_FIGURES") == "1":
        print("Saved", save_campaign_figure(figure, ARTIFACT_ROOT,
                                             "balanced_v1", "ddr_full", "error_gallery"))
else:
    print("PENDING gallery: test images remain unopened; opt in only after final evaluation is frozen")

PENDING gallery: test images remain unopened; opt in only after final evaluation is frozen


**Interpretasi dan keterbatasan.** Runner baru menyimpan image ID, label aktual/prediksi, confidence, serta SHA-256 di hasil final test **hanya ketika perintah test eksplisit kelak dijalankan**. `plot_error_gallery(..., allow_test_images=True)` tersedia untuk tahap pelaporan setelah evaluasi dikunci; jangan memilih contoh berdasarkan test untuk training/tuning. JSON historis tanpa prediksi per gambar tetap PENDING.
**Implementation assumption:** Galeri FP/FN memproyeksikan grade 0 ke No DR dan grade 1–4 ke DR. Kesalahan grade lima kelas tetap dilaporkan terpisah; proyeksi ini tidak diklaim sebagai protokol Figure 7 penulis.


**DDR recovery 80 epoch:** ringkasan final test dan confusion matrix agregat kini tersedia pada bagian berikut. Prediksi per gambar tetap berada di server; galeri kesalahan lokal belum tersedia. Sel di atas hanya untuk `balanced_v1` historis dan tidak dipakai untuk mengklaim hasil recovery. Tidak ada contoh gambar kesalahan yang ditampilkan.

## DDR recovery 80 epoch — analisis kesalahan agregat final test

`DDR_FINAL_TEST` mencatat **277** kesalahan dari **1.252** gambar, berdasarkan [`final_test_summary.json`](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/final_test_summary.json). Grade **1** mempunyai F1 terendah (**0,323144**). Grade **3** memiliki support **23** dan hanya **11** prediksi benar (recall **0,478261**), sehingga estimasinya tidak pasti. Kebingungan grade **2→3** berjumlah **16**, sedangkan **3→2** berjumlah **11**. Kesalahan paling banyak secara absolut adalah **0→1: 74** dan **2→1: 52**. Matriks agregat tidak menjelaskan penyebab klinis atau ciri visual kesalahan tersebut.

Sel read-only berikut menghitung ulang jumlah kesalahan dan pasangan grade dari matriks asli. Prediksi per gambar sengaja tetap pada server GPU; tidak ada contoh citra atau galeri kesalahan lokal. Sel galeri `balanced_v1` historis di atas tetap terpisah dan memerlukan opt-in tersendiri.

In [4]:
# Aggregate-only DDR recovery error analysis; no test images or inference.
import json
from pathlib import Path
summary_path = ROOT / "results/DDR/full_oversampling_fp32_math_mb8_80epochs/final_test_summary.json"
if not summary_path.is_file():
    print("PENDING — DDR final-test summary is unavailable")
else:
    result = json.loads(summary_path.read_text(encoding="utf-8"))
    cm = result["confusion_matrix"]
    if (result.get("dataset") != "DDR" or result.get("split") != "test"
            or result.get("campaign") != "cadvit_tables_1_5_balanced_fp32_math_mb8_diag"
            or len(cm) != 5 or any(len(row) != 5 for row in cm)
            or sum(map(sum, cm)) != result["test_samples"]):
        raise ValueError("DDR final-test matrix or provenance is inconsistent")
    errors = sorted(((cm[i][j], i, j) for i in range(5) for j in range(5)
                     if i != j and cm[i][j]), reverse=True)
    print("DDR_FINAL_TEST; errors:", sum(count for count, _, _ in errors))
    print("Largest confusions (count, true, predicted):", errors[:5])
    for item in result["per_class"]:
        print("Grade", item["grade"], "F1", item["f1"], "recall", item["recall"],
              "support", item["support"])
    print("Image-level examples: unavailable locally; predictions remain on the server")